# Stage 2: train YOLO on D-Fire (Kaggle GPU)

**Before you run anything** (right-hand panel of the Kaggle notebook):
1. **Add Input**: search for the D-Fire dataset and add it.
2. **Session options → Accelerator**: choose **GPU T4 x2** (or P100).
3. **Session options → Internet**: on (needed to install Ultralytics and download the starting model).

Then run the cells **one at a time**. Leave `QUICK = True` for the first run (about 5 minutes) to check that everything works.
After that, set `QUICK = False` and use **Save Version → Save & Run All** so the long training keeps running even if you close the tab.

What you take home at the end: `best.pt` (the trained model), from the **Output** tab.

## 1. Settings

In [ ]:
QUICK = True        # True = 5-minute check on a small random sample. False = the real training run.

MODEL   = 'yolo26n.pt'   # small and fast, so it can also run live on the laptop later (Stage 4)
IMGSZ   = 640
EPOCHS  = 1 if QUICK else 60
BATCH   = 32
VAL_SHARE = 0.15         # share of the training images kept aside for validation
SEED    = 42

## 2. Install YOLO

In [ ]:
%pip install -q ultralytics
import ultralytics, torch
print('Ultralytics', ultralytics.__version__, '| torch', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - turn on the GPU in Session options')

## 3. Find the dataset
Looks inside `/kaggle/input` for folders named `images` with a `labels` folder next to them, and prints what it found.

In [ ]:
import os, random
from pathlib import Path
from collections import Counter

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def count_files(d, exts=None):
    return sum(1 for p in d.iterdir() if p.is_file() and (exts is None or p.suffix.lower() in exts))

found = {}
for img_dir in sorted(Path('/kaggle/input').rglob('images')):
    lab_dir = img_dir.parent / 'labels'
    if img_dir.is_dir() and lab_dir.is_dir():
        split = img_dir.parent.name.lower()
        n_img, n_lab = count_files(img_dir, IMG_EXT), count_files(lab_dir, {'.txt'})
        print(f'{img_dir}  ->  {n_img} images, {n_lab} label files')
        found[split] = img_dir

if not found:
    raise SystemExit('No images/ + labels/ folders found. Did you add the D-Fire dataset with "Add Input"?')

TRAIN_DIR = found.get('train')
VAL_DIR   = found.get('val') or found.get('valid') or found.get('validation')
TEST_DIR  = found.get('test')
print('\ntrain:', TRAIN_DIR, '\nval  :', VAL_DIR, '\ntest :', TEST_DIR)
assert TRAIN_DIR is not None and TEST_DIR is not None, 'Need at least a train and a test folder. Tell Claude what was printed above.'

## 3b. Copy the dataset to the fast local disk (full run only)
Kaggle's input folder is a slow network drive and read-only. Reading from it made each epoch wait for the photos, and YOLO had to skip a few slightly damaged photos it could not repair there.
Copying once to the notebook's own disk (about 10 minutes) makes every epoch much faster and lets YOLO repair those photos. In QUICK mode this is skipped.


In [ ]:
import shutil

if not QUICK:
    LOCAL = Path('/tmp/dfire_local')     # the notebook's own disk; not saved as output
    free_gb = shutil.disk_usage('/tmp').free / 1e9
    print(f'Free space on local disk: {free_gb:.0f} GB')
    for name, d in [('train', TRAIN_DIR), ('val', VAL_DIR), ('test', TEST_DIR)]:
        if d is None:
            continue
        dest = LOCAL / name
        if not (dest / 'images').exists():
            print(f'Copying {name} ...', flush=True)
            shutil.copytree(d.parent / 'images', dest / 'images')
            shutil.copytree(d.parent / 'labels', dest / 'labels')
        print(f'{name}: {count_files(dest / "images", IMG_EXT)} images copied')
    TRAIN_DIR = LOCAL / 'train' / 'images'
    VAL_DIR   = (LOCAL / 'val' / 'images') if VAL_DIR is not None else None
    TEST_DIR  = LOCAL / 'test' / 'images'
    print('\nNow using the local copy:', TRAIN_DIR.parent.parent)
else:
    print('QUICK mode: using the dataset where it is (no copy).')


## 4. Check the labels mean 0 = smoke, 1 = fire
D-Fire's published counts: **1,164** fire-only images, **5,867** smoke-only, **4,658** both, **9,838** none.
If this copy has them the other way round, the cell stops so the names don't get swapped.

In [ ]:
def image_kind(label_file):
    ids = set()
    if label_file.exists():
        for line in label_file.read_text().splitlines():
            parts = line.split()
            if parts:
                ids.add(int(float(parts[0])))
    if ids == set():   return 'none'
    if ids == {0}:     return 'only 0'
    if ids == {1}:     return 'only 1'
    if ids == {0, 1}:  return 'both'
    return 'other ids: ' + str(sorted(ids))

kinds = Counter()
for d in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    if d is None:
        continue
    for img in d.iterdir():
        if img.suffix.lower() in IMG_EXT:
            kinds[image_kind(d.parent / 'labels' / (img.stem + '.txt'))] += 1
for k, v in sorted(kinds.items()):
    print(f'{k:<12}{v:>7}')

assert not any(k.startswith('other') for k in kinds), 'Label ids other than 0 and 1 found: tell Claude.'
if kinds['only 1'] > kinds['only 0']:
    raise SystemExit('Looks like 0 = fire and 1 = smoke in this copy (the reverse of D-Fire). Stop and tell Claude.')
NAMES = {0: 'smoke', 1: 'fire'}
print('\nOK: 0 = smoke, 1 = fire')

## 5. Make the train / validation / test lists
D-Fire has no validation set, so 15% of the training images are set aside.
Many images are frames from the same videos, and neighbouring frames look almost identical.
So images are kept aside **in blocks of 50 consecutive file names**, not one by one: this keeps near-copies of a validation image out of training.
Nothing is copied: the lists just point to the images where they are.

In [ ]:
WORK = Path('/kaggle/working')
rng = random.Random(SEED)

def images_in(d):
    return sorted(p for p in d.iterdir() if p.suffix.lower() in IMG_EXT)

train_imgs = images_in(TRAIN_DIR)
test_imgs  = images_in(TEST_DIR)

if VAL_DIR is not None:
    val_imgs = images_in(VAL_DIR)
    print('Using the validation folder that came with this copy of the dataset.')
else:
    blocks = [train_imgs[i:i + 50] for i in range(0, len(train_imgs), 50)]
    rng.shuffle(blocks)
    n_val_blocks = round(len(blocks) * VAL_SHARE)
    val_imgs   = sorted(p for b in blocks[:n_val_blocks] for p in b)
    train_imgs = sorted(p for b in blocks[n_val_blocks:] for p in b)

if QUICK:   # a RANDOM small sample (not the first files alphabetically, which are mostly empty)
    train_imgs = rng.sample(train_imgs, min(1500, len(train_imgs)))
    val_imgs   = rng.sample(val_imgs, min(300, len(val_imgs)))

for name, imgs in [('train', train_imgs), ('val', val_imgs), ('test', test_imgs)]:
    (WORK / f'{name}.txt').write_text('\n'.join(str(p) for p in imgs) + '\n')
    print(f'{name:<6}{len(imgs):>7} images')

yaml_text = f"""path: {WORK}
train: {WORK}/train.txt
val: {WORK}/val.txt
test: {WORK}/test.txt
names:
  0: {NAMES[0]}
  1: {NAMES[1]}
"""
(WORK / 'dfire_kaggle.yaml').write_text(yaml_text)
print('\n' + yaml_text)

## 6. Train
QUICK mode: about 5 minutes. Full run: a few hours (it stops early if validation stops improving).
For the full run use **Save Version → Save & Run All**, so it runs in the background.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
model.train(
    data=str(WORK / 'dfire_kaggle.yaml'),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=15,          # stop if validation mAP has not improved for 15 epochs
    seed=SEED,
    device=0,
    workers=4,
    project=str(WORK / 'runs'),
    name='quick_test' if QUICK else 'dfire_yolo26n',
    exist_ok=True,
)

## 7. Test on the official test images
These images were never used for training or for choosing the best epoch, so this is the honest score.
- **mAP50**: how well boxes match the real fire/smoke (a box counts if it overlaps the true one by at least half). 1.0 is perfect.
- **mAP50-95**: the same, but also rewards very precise boxes. Always lower than mAP50.
- **Precision**: of the boxes the model drew, how many were real fire/smoke.
- **Recall**: of the real fire/smoke, how much the model found.

In [ ]:
run_dir = WORK / 'runs' / ('quick_test' if QUICK else 'dfire_yolo26n')
best = YOLO(str(run_dir / 'weights' / 'best.pt'))
metrics = best.val(data=str(WORK / 'dfire_kaggle.yaml'), split='test', imgsz=IMGSZ, batch=BATCH,
                   project=str(WORK / 'runs'), name=run_dir.name + '_test', exist_ok=True)

print('\nTEST RESULTS')
print(f'mAP50     : {metrics.box.map50:.3f}')
print(f'mAP50-95  : {metrics.box.map:.3f}')
print(f'Precision : {metrics.box.mp:.3f}')
print(f'Recall    : {metrics.box.mr:.3f}')
for i, name in NAMES.items():
    p, r, ap50, ap = metrics.box.class_result(i)
    print(f'  {name:<6} precision {p:.3f}  recall {r:.3f}  mAP50 {ap50:.3f}  mAP50-95 {ap:.3f}')

## 8. Look at some predictions

In [ ]:
import matplotlib.pyplot as plt

sample = rng.sample(test_imgs, 8)
plt.figure(figsize=(16, 8))
for i, img in enumerate(sample):
    r = best.predict(str(img), imgsz=IMGSZ, conf=0.25, verbose=False)[0]
    plt.subplot(2, 4, i + 1)
    plt.imshow(r.plot()[:, :, ::-1])   # plot() gives BGR; flip to RGB
    plt.axis('off')
    plt.title(img.name, fontsize=8)
plt.tight_layout(); plt.show()

## 9. Collect the files to download
After the run, open the **Output** tab of this notebook version and download:
- `best.pt`: the trained model. Put it in your project folder on the laptop (it is ignored by git).
- `results.csv` and the `.png` plots: training curves, confusion matrix, precision-recall curves for your report.

In [ ]:
import shutil
out = WORK / 'to_download'
out.mkdir(exist_ok=True)
shutil.copy(run_dir / 'weights' / 'best.pt', out / 'best.pt')
for f in list(run_dir.glob('*.png')) + list(run_dir.glob('*.csv')) + list((WORK / 'runs' / (run_dir.name + '_test')).glob('*.png')):
    shutil.copy(f, out / (('test_' if f.parent.name.endswith('_test') else '') + f.name))
for f in sorted(out.iterdir()):
    print(f.name)